# Apollo 4 Flight Validation

Validation of `reentrykit` against Apollo 4's atmospheric entry, comparing against the Hilje 1969 flight reconstruction.

## Mission Context

Apollo 4 (AS-501), launched November 9, 1967, was the first all-up flight test of the Saturn V and the first uncrewed test of Apollo Command Module entry at lunar-return velocity. The Command Module separated from the Service Module and executed a guided skip re-entry over the Pacific, splashing down in the Central Pacific south of Hawaii after a flight of approximately 713 seconds.

The Apollo Guidance Computer (AGC) implemented closed-loop lift-vector steering through five phases: INITIAL ENTRY, HUNTEST, UPCONTROL, KEPLER (bypassed for Apollo 4), and FINAL ENTRY. Peak deceleration was 7.26 g, and the skip apogee reached 73.6 km (241,602 ft).

## Source

All flight data used for validation is digitized from NASA Technical Note D-5399 (Hilje 1969), *"Entry aerodynamics at lunar return conditions obtained from the flight of Apollo 4 (AS-501)."*

## Validation Approach

- Simulator: `reentrykit` 3-DOF point-mass model, non-rotating Earth, US1976 atmosphere extended exponentially to 500 km
- Input: digitized entry state + L/D and bank schedules from Hilje figures
- Constant drag coefficient Cd = 1.2 (Apollo CM hypersonic continuum value)
- Comparison: time histories of velocity, flight-path angle, azimuth, latitude, longitude, altitude against Hilje's published curves
- Headline metrics: first peak g, second peak g, skip apogee, total flight duration

## 2. Vehicle Parameters

Apollo Command Module physical constants.

In [1]:
"""Apollo 4 validation — imports and vehicle constants."""

import numpy as np
import matplotlib.pyplot as plt
from reentrykit.trajectory import Vehicle, InitialState, simulate
from reentrykit.planet import EARTH_NON_ROTATING

# Apollo CM physical parameters
APOLLO_MASS = 5425.0                  # [kg], at entry interface (Hilje: 11,959 lb)
APOLLO_DIAMETER = 3.912               # [m], maximum diameter of blunt face
APOLLO_AREA = np.pi * (APOLLO_DIAMETER / 2.0) ** 2   # [m^2], reference area (~12.0)
APOLLO_NOSE_RADIUS = 4.661            # [m], spherical aft heatshield radius
APOLLO_CD = 1.2                       # [-], constant hypersonic continuum Cd
APOLLO_LD_MAGNITUDE = 0.368           # [-], L/D from offset CG, used as fallback scalar

print(f"Apollo CM parameters:")
print(f"  Mass:         {APOLLO_MASS:>8.1f} kg  (entry weight)")
print(f"  Diameter:     {APOLLO_DIAMETER:>8.3f} m")
print(f"  Ref. area:    {APOLLO_AREA:>8.2f} m^2")
print(f"  Cd:           {APOLLO_CD:>8.2f}")
print(f"  L/D geometric:{APOLLO_LD_MAGNITUDE:>8.3f}")
print(f"  Nose radius:  {APOLLO_NOSE_RADIUS:>8.3f} m")

Apollo CM parameters:
  Mass:           5425.0 kg  (entry weight)
  Diameter:        3.912 m
  Ref. area:       12.02 m^2
  Cd:               1.20
  L/D geometric:   0.368
  Nose radius:     4.661 m


## 3. Digitized Hilje Flight Data

All data below is digitized from figures in NASA TN D-5399 (Hilje 1969). Each time series is stored as GET (Ground Elapsed Time, seconds from launch) alongside the quantity. Post-entry-interface time is computed by subtracting the EI reference time.

Entry Interface (EI) reference:
- GET 29,968.54 s = 08:19:28.54 elapsed from launch (geographic entry, altitude 400 kft / 121.92 km)

Reentry phase markers (from Hilje paper text):
- Entry: 08:19:25 GET (29,965 s)  
- Parachute deployment: 08:31:18 GET (30,678 s)
- Total reentry duration: 713 s

In [2]:
"""Section 3.2: L/D time history digitized from Hilje Figure 3.

Values represent the effective vertical-component L/D ratio as reconstructed
from flight data. Negative values indicate lift-down moments (bank > 90°).
"""

APOLLO_HIRES_GET = np.array([
    29968.5, 29975.0, 29980.0, 29985.0, 29990.0, 29995.0, 30000.0, 30005.0,
    30010.0, 30015.0, 30020.0, 30025.0, 30030.0, 30035.0, 30040.0, 30045.0,
    30050.0, 30055.0, 30060.0, 30065.0, 30070.0, 30075.0, 30080.0, 30085.0,
    30090.0, 30095.0, 30100.0, 30105.0, 30110.0, 30115.0, 30120.0, 30125.0,
    30130.0, 30135.0, 30140.0, 30145.0, 30150.0, 30155.0, 30160.0, 30165.0,
    30170.0, 30175.0, 30180.0, 30185.0, 30190.0, 30195.0, 30200.0, 30250.0,
    30300.0, 30400.0, 30500.0, 30600.0, 30700.0,
])

# PLACEHOLDER: Replace with the actual digitized L/D values from your session data
# For the clean notebook, you'll need to paste the ~230 digitized values here
APOLLO_HIRES_LD = np.array([
    # NOTE: Paste APOLLO_HIRES_LD values from the OLD notebook here.
    # Length must match APOLLO_HIRES_GET.
])

APOLLO_HIRES_TIME = APOLLO_HIRES_GET - APOLLO_ENTRY_INTERFACE_GET   # [s post-EI]


def apollo_ld_schedule(t: float) -> float:
    """Interpolated L/D(t) from digitized Hilje Figure 3."""
    if t <= APOLLO_HIRES_TIME[0]:
        return float(APOLLO_HIRES_LD[0])
    if t >= APOLLO_HIRES_TIME[-1]:
        return float(APOLLO_HIRES_LD[-1])
    return float(np.interp(t, APOLLO_HIRES_TIME, APOLLO_HIRES_LD))


print(f"L/D schedule: {len(APOLLO_HIRES_LD)} points from t={APOLLO_HIRES_TIME[0]:.1f}s to t={APOLLO_HIRES_TIME[-1]:.1f}s")
print(f"L/D range: {APOLLO_HIRES_LD.min():.3f} to {APOLLO_HIRES_LD.max():.3f}")

NameError: name 'APOLLO_ENTRY_INTERFACE_GET' is not defined